In [0]:
from pyspark.sql import functions as F

# Date range taken from the orders
bounds = (spark.read.table("databricks_catalog.silver_schema.orders")
          .select(F.min("order_date").alias("start"), F.max("order_date").alias("end"))
          .first())

dim_date = (
    spark.createDataFrame([(bounds["start"], bounds["end"])], ["start", "end"])
    .select(F.explode(F.sequence("start", "end", F.expr("interval 1 day"))).alias("date"))
    .select(
        "date",
        F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
        F.year("date").alias("year"),
        F.quarter("date").alias("quarter"),
        F.month("date").alias("month"),
        F.date_format("date", "MMMM").alias("month_name"),
        F.weekofyear("date").alias("week_of_year"),
        F.dayofmonth("date").alias("day"),
        F.date_format("date", "EEEE").alias("day_name"),
        F.dayofweek("date").isin(1, 7).alias("is_weekend"),
    )
    .withColumn("loaded_at", F.current_timestamp())
)

(dim_date.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("databricks_catalog.gold_schema.dim_date"))

print("✓ dim_date written:", dim_date.count(), "rows")

✓ dim_date written: 731 rows


In [0]:
%sql
SELECT COUNT(*) AS days, MIN(date) AS first_day, MAX(date) AS last_day
FROM databricks_catalog.gold_schema.dim_date;

days,first_day,last_day
731,2023-01-01,2024-12-31
